# 🕊️ Fine-tune your Christian AI — in Colab (newest models work here)

This trains a LoRA on the model **you** pick, using the `train.jsonl` you
already made. Unlike AutoTrain, Colab is a fresh machine, so we install
the **latest** libraries (transformers v5) — which means the **newest
models (Gemma 4, Qwen 3.x) actually load here.**

## Steps
1. **Runtime → Change runtime type → GPU.** (T4 is free and fine for models
   up to ~8B. For 27B-32B you need an A100 — Runtime type → A100, Colab Pro.)
2. **STEP 1**: paste your HF token + pick your model.
3. **STEP 2**: upload your `train.jsonl` when prompted.
4. **Runtime → Run all.** Watch the loss drop. It auto-pushes to your HF account.

> 💡 Model vs GPU (4-bit QLoRA): a **T4 (16GB)** fits up to ~8-9B. An
> **A100 (40GB)** fits up to ~32B. Pick accordingly — the notebook prints
> your GPU so you know.

In [ ]:
#@title STEP 0 — install the latest libraries (~2-3 min)
# Latest of everything = mutually compatible AND supports the newest models.
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets huggingface_hub
print("\n✅ installed. (If you see red dependency warnings, ignore them — they're harmless.)")

In [ ]:
#@title ✏️ STEP 1 — your token, your model, your output name

MY_HF_TOKEN = "PASTE-YOUR-WRITE-TOKEN"  #@param {type:"string"}

# Pick ANY model — Colab's fresh libraries load the newest ones.
#   Fits a free T4 (~8B):  Qwen/Qwen3-8B  |  google/gemma-4-E4B-it  |  Qwen/Qwen2.5-7B-Instruct
#   Needs an A100 (27-32B): Qwen/Qwen2.5-32B-Instruct  |  google/gemma-4-31B-it
MODEL = "Qwen/Qwen3-8B"  #@param {type:"string"}

OUTPUT_REPO = "moonshineai/solideo-v1"  #@param {type:"string"}

In [ ]:
#@title 📤 STEP 2 — upload your train.jsonl (click 'Choose Files')
from google.colab import files
print("Pick the train.jsonl you downloaded earlier:")
up = files.upload()
assert "train.jsonl" in up, "Please upload the file named exactly train.jsonl"
print("✅ got train.jsonl")

In [ ]:
#@title ▶️ STEP 3 — train (Runtime → Run all). Watch the loss go down.
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

assert MY_HF_TOKEN and "PASTE" not in MY_HF_TOKEN, "Paste your WRITE token in STEP 1 first!"
login(token=MY_HF_TOKEN)

assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → GPU, then Run all again."
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu}  ({vram:.0f} GB)")
print(f"Model: {MODEL}\n")

ds = load_dataset("json", data_files="train.jsonl", split="train")
print(f"Training examples: {len(ds)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL)

# 4-bit QLoRA so big models fit on a small GPU
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16,
)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM", target_modules="all-linear",
)

cfg = SFTConfig(
    output_dir="sdg-out",
    per_device_train_batch_size=2,      # lower to 1 if you hit out-of-memory
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    max_length=2048,                    # if this errors, rename to max_seq_length
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    push_to_hub=True,
    hub_model_id=OUTPUT_REPO,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=ds,                   # has a 'messages' column -> chat template auto-applied
    peft_config=peft_config,
    processing_class=tokenizer,         # if this errors, rename to tokenizer=tokenizer
)
trainer.train()
trainer.push_to_hub()
print(f"\n✅ DONE → https://huggingface.co/{OUTPUT_REPO}")

## 🆘 If it errors (and how to read it)

Errors print **right in the cell** — no hidden logs. Common ones:

| Error | Fix |
|---|---|
| `No GPU!` | Runtime → Change runtime type → **GPU** → Run all again |
| `CUDA out of memory` | Your model is too big for this GPU. Use a smaller `MODEL` (e.g. `Qwen/Qwen3-8B`) or set `per_device_train_batch_size=1`, or switch to an A100 (Colab Pro) |
| `gated` / `401` / `403` on the model | Accept its license on its HF page, and make sure your token has access |
| `SFTConfig got unexpected keyword 'max_length'` | change `max_length=2048` → `max_seq_length=2048` |
| `SFTTrainer got unexpected keyword 'processing_class'` | change `processing_class=tokenizer` → `tokenizer=tokenizer` |
| token / push error | Your token must be a **Write** token (huggingface.co/settings/tokens) |

Copy the **last ~15 lines** of any error and send them to Claude.

*Soli Deo Gloria.*